# BioMini Phase I Educational Materials

## Notebook 02 — Inheritance and Sequence Abstraction
### Inheritance에서 Sequence Abstraction으로

Notebook 01에서는 `Molecule`, `Protein`, `AminoAcid`를 직접 만들었습니다.

이번 Notebook에서는 DNA, RNA, Protein에 반복되는 sequence 관련 책임을 찾아
`SequenceMolecule`이라는 공통 abstraction으로 추출합니다.

이 Notebook은 BioMini Phase I의 **Step 2 — SequenceMolecule Abstraction**과
**Step 3 — DNA / RNA / Protein Hierarchy**를 교육용으로 재구성합니다.

---

## Learning Objectives

이 Notebook을 마치면 다음을 할 수 있어야 합니다.

1. inheritance와 abstraction의 차이를 설명한다.
2. 여러 class에서 반복되는 책임을 찾아 superclass로 추출한다.
3. `SequenceMolecule`의 필요성을 설명한다.
4. `NucleicAcid`, `DNA`, `RNA`, `Protein` hierarchy를 설계한다.
5. subclass-specific validation을 구현한다.
6. normalization과 biological validation을 분리해서 생각한다.
7. inheritance가 언제 유용하고 언제 과도해질 수 있는지 설명한다.

# 1. Starting Point — DNA, RNA, Protein을 각각 만들면?

Notebook 01의 방식대로라면 DNA, RNA, Protein을 각각 별도의 class로 만들 수 있습니다.

먼저 가장 단순한 형태를 일부러 중복해서 작성해 봅니다.

In [ ]:
class ValidationError(ValueError):
    pass


class Molecule:
    def __init__(self, name: str):
        if not isinstance(name, str) or not name.strip():
            raise ValidationError("name must be a non-empty string")
        self.name = name.strip()

    def get_name(self) -> str:
        return self.name

In [ ]:
class DNA(Molecule):
    def __init__(self, name: str, sequence: str):
        super().__init__(name)

        if not isinstance(sequence, str) or not sequence.strip():
            raise ValidationError("sequence must be a non-empty string")

        self.sequence = "".join(sequence.split()).upper()

    def get_sequence(self) -> str:
        return self.sequence

    def __len__(self) -> int:
        return len(self.sequence)


class RNA(Molecule):
    def __init__(self, name: str, sequence: str):
        super().__init__(name)

        if not isinstance(sequence, str) or not sequence.strip():
            raise ValidationError("sequence must be a non-empty string")

        self.sequence = "".join(sequence.split()).upper()

    def get_sequence(self) -> str:
        return self.sequence

    def __len__(self) -> int:
        return len(self.sequence)


class Protein(Molecule):
    def __init__(self, name: str, sequence: str):
        super().__init__(name)

        if not isinstance(sequence, str) or not sequence.strip():
            raise ValidationError("sequence must be a non-empty string")

        self.sequence = "".join(sequence.split()).upper()

    def get_sequence(self) -> str:
        return self.sequence

    def __len__(self) -> int:
        return len(self.sequence)

In [ ]:
dna = DNA("dna1", " atgc ")
rna = RNA("rna1", " augc ")
protein = Protein("protein1", " acde ")

print(dna.get_sequence(), len(dna))
print(rna.get_sequence(), len(rna))
print(protein.get_sequence(), len(protein))

이 코드는 작동하지만 세 class에 거의 같은 코드가 반복됩니다.

```text
sequence type check
sequence empty check
whitespace removal
uppercase normalization
get_sequence()
__len__()
```

이런 중복은 단순한 코드 중복 이상의 의미를 가집니다.

공통 책임이 여러 class에 흩어져 있다는 뜻입니다.

# 2. 공통 책임 찾기

DNA, RNA, Protein은 생물학적으로 서로 다른 분자입니다.

하지만 software 관점에서는 모두 다음 공통 특성을 가집니다.

```text
ordered sequence를 가진다
sequence를 normalization한다
sequence length를 계산한다
sequence를 반환한다
```

따라서 공통 computational abstraction을 만들 수 있습니다.

```text
Molecule
   ↓
SequenceMolecule
```

In [ ]:
class SequenceMolecule(Molecule):
    def __init__(self, name: str, sequence: str):
        super().__init__(name)

        if not isinstance(sequence, str):
            raise ValidationError("sequence must be a string")

        normalized = "".join(sequence.split()).upper()

        if not normalized:
            raise ValidationError("sequence must be a non-empty string")

        self.sequence = normalized

    def get_sequence(self) -> str:
        return self.sequence

    def __len__(self) -> int:
        return len(self.sequence)

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"name='{self.name}', length={len(self)})"
        )

이제 DNA, RNA, Protein은 공통 sequence logic을 상속받을 수 있습니다.

In [ ]:
class DNA(SequenceMolecule):
    pass


class RNA(SequenceMolecule):
    pass


class Protein(SequenceMolecule):
    pass

In [ ]:
dna = DNA("dna1", " atgc ")
rna = RNA("rna1", " augc ")
protein = Protein("protein1", " acde ")

print(dna)
print(rna)
print(protein)

print(dna.get_sequence())
print(rna.get_sequence())
print(protein.get_sequence())

여기까지의 설계는 중복을 줄였지만 아직 중요한 문제가 남아 있습니다.

현재는 다음도 허용됩니다.

```python
DNA("bad", "AUGC")
RNA("bad", "ATGC")
Protein("bad", "ATGC")
```

즉 sequence의 **형식적 처리**는 공통화했지만,
각 생물학적 class의 **alphabet rule**은 아직 없습니다.

# 3. Biological Alphabet Validation

각 subclass는 자기 자신의 biological alphabet을 책임지게 합니다.

- DNA: `A`, `T`, `G`, `C`
- RNA: `A`, `U`, `G`, `C`
- Protein: canonical 20 amino acids

공통 sequence handling은 superclass에,
domain-specific validation은 subclass에 둡니다.

In [ ]:
class DNA(SequenceMolecule):
    valid_codes = set("ATGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid DNA code(s): {sorted(invalid_codes)}"
            )


class RNA(SequenceMolecule):
    valid_codes = set("AUGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid RNA code(s): {sorted(invalid_codes)}"
            )

Protein alphabet도 정의합니다.

Notebook 01에서 만들었던 `AminoAcid`의 canonical 20-letter alphabet을 재사용합니다.

In [ ]:
class AminoAcid:
    acceptable_codes = set("ARNDCQEGHILKMFPSTWYV")


class Protein(SequenceMolecule):
    valid_codes = AminoAcid.acceptable_codes

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid protein code(s): {sorted(invalid_codes)}"
            )

In [ ]:
dna = DNA("dna1", "atgc")
rna = RNA("rna1", "augc")
protein = Protein("protein1", "acde")

print(dna)
print(rna)
print(protein)

## Invalid Input 확인

In [ ]:
examples = [
    ("DNA", DNA, "AUGC"),
    ("RNA", RNA, "ATGC"),
    ("Protein", Protein, "ACDX"),
]

for label, cls, sequence in examples:
    try:
        cls("bad", sequence)
    except ValidationError as exc:
        print(f"{label:7s} -> {exc}")

# 4. NucleicAcid Abstraction

DNA와 RNA는 Protein보다 서로 더 가까운 공통 개념을 공유합니다.

```text
SequenceMolecule
   ├── DNA
   ├── RNA
   └── Protein
```

도 가능하지만, biological domain을 더 명확하게 표현하려면 다음이 자연스럽습니다.

```text
SequenceMolecule
   ├── NucleicAcid
   │    ├── DNA
   │    └── RNA
   └── Protein
```

현재 단계에서 `NucleicAcid`가 별도 동작을 많이 갖지 않더라도,
DNA/RNA가 같은 biological category임을 architecture에 표현할 수 있습니다.

In [ ]:
class NucleicAcid(SequenceMolecule):
    """Base class for nucleic-acid sequence objects."""
    pass

In [ ]:
class DNA(NucleicAcid):
    valid_codes = set("ATGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid DNA code(s): {sorted(invalid_codes)}"
            )


class RNA(NucleicAcid):
    valid_codes = set("AUGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid RNA code(s): {sorted(invalid_codes)}"
            )

# 5. 최종 Educational Hierarchy

이번 Notebook에서 만든 구조는 다음과 같습니다.

```text
Molecule
   ↓
SequenceMolecule
   ├── NucleicAcid
   │    ├── DNA
   │    └── RNA
   └── Protein
```

각 layer의 책임은 다음과 같습니다.

| Class | Responsibility |
|---|---|
| `Molecule` | 공통 molecule identity |
| `SequenceMolecule` | sequence 저장, normalization, 길이 |
| `NucleicAcid` | DNA/RNA의 공통 biological category |
| `DNA` | DNA alphabet validation |
| `RNA` | RNA alphabet validation |
| `Protein` | protein alphabet validation |

# 6. Normalization과 Validation은 다른 책임

예를 들어 입력:

```text
" a t g c "
```

는 normalization을 거쳐:

```text
"ATGC"
```

가 됩니다.

그 후 DNA alphabet validation이 수행됩니다.

즉:

```text
raw input
   ↓
normalization
   ↓
canonical internal representation
   ↓
domain-specific validation
```

이 순서를 분리해서 생각하는 것이 중요합니다.

In [ ]:
dna = DNA(
    "normalized_dna",
    " a t g c "
)

print(dna.get_sequence())

# 7. Self-Check

In [ ]:
dna = DNA("dna", "ATGC")
rna = RNA("rna", "AUGC")
protein = Protein("protein", "ACDE")

assert dna.get_sequence() == "ATGC"
assert rna.get_sequence() == "AUGC"
assert protein.get_sequence() == "ACDE"

assert len(dna) == 4
assert len(rna) == 4
assert len(protein) == 4

assert isinstance(dna, NucleicAcid)
assert isinstance(rna, NucleicAcid)
assert isinstance(protein, SequenceMolecule)

print("Notebook 02 self-check: PASS")

# 8. Inheritance가 모든 문제의 답은 아니다

이번 Notebook에서는 inheritance가 잘 맞았습니다.

왜냐하면 관계가 자연스럽기 때문입니다.

```text
DNA is-a NucleicAcid
RNA is-a NucleicAcid
Protein is-a SequenceMolecule
```

하지만 앞으로 분석 기능까지 모두 inheritance로 해결하려고 하면 문제가 생깁니다.

예를 들어:

```text
Protein
   ↓
ProteinAnalyzer ?
```

`ProteinAnalyzer`는 Protein 자체가 아닙니다.

분석기는 Protein을 **가지고(has-a)** 분석합니다.

이 차이는 Notebook 04에서 중요하게 다룹니다.

# 9. Empty Sequence에 대한 질문

현재 educational `SequenceMolecule`은 empty sequence를 모두 거부합니다.

```python
Protein("empty", "")
```

도 실패합니다.

하지만 실제 BioMini Phase I에서는 나중에 중요한 edge case가 발견되었습니다.

예:

```text
RNA = UAA
translation
   ↓
Protein = ""
```

첫 codon이 stop codon이면 computationally valid한 empty translation product가 나올 수 있습니다.

따라서 canonical BioMini에서는 최종적으로:

```text
DNA     → empty 불허
RNA     → empty 불허
Protein → valid translation result인 경우 empty 허용
```

이라는 정책을 채택했습니다.

여기서는 이 문제를 아직 구현하지 않고,
'validation rule도 scientific semantics에 따라 바뀔 수 있다'는 점만 기억합니다.

# 10. Canonical BioMini와 비교

실제 BioMini Phase I의 canonical hierarchy는 다음과 같습니다.

```text
DomainEntity
   ↓
Molecule
   ↓
SequenceMolecule
   ├── NucleicAcid
   │    ├── DNA
   │    └── RNA
   └── Protein
```

교육용 구현과의 차이는 `DomainEntity`입니다.

production code에서는 각 object가 다음을 추가로 가집니다.

```text
stable ID
metadata
provenance
schema version
```

하지만 이 책임들은 Notebook 07에서 필요성이 생길 때 다룹니다.

In [ ]:
try:
    from biomini import DNA as BioMiniDNA
    from biomini import RNA as BioMiniRNA
    from biomini import Protein as BioMiniProtein

    bm_dna = BioMiniDNA("dna1", "ATGC")
    bm_rna = BioMiniRNA("rna1", "AUGC")
    bm_protein = BioMiniProtein("protein1", "ACDE")

    print("Canonical DNA     :", bm_dna)
    print("Canonical RNA     :", bm_rna)
    print("Canonical Protein :", bm_protein)

except ImportError:
    print(
        "BioMini package is not installed in this environment.\n"
        "From the repository root, run:\n"
        "    python -m pip install -e ."
    )

# 11. Educational Code vs Canonical Code

이번 Notebook도 production source를 그대로 복사하지 않습니다.

```text
Notebook
= abstraction이 왜 필요한지 이해하기 위한 최소 구현

biomini/
= 현재 프로젝트의 canonical implementation
```

학생은 먼저 반복 문제를 직접 경험하고,
그 뒤 실제 BioMini architecture와 연결합니다.

# 12. Design Question — 다음 문제

이제 DNA와 RNA가 있습니다.

그렇다면 DNA의 complement와 transcription,
RNA의 translation은 어디에 둘까요?

예:

```text
DNA
 ├── complement()
 ├── reverse_complement()
 └── transcribe()

RNA
 └── translate()
```

다음 Notebook에서는 sequence object가 단순히 데이터를 저장하는 수준을 넘어
**biological transformation을 수행하는 domain object**로 발전합니다.

```text
DNA
 ↓ transcription
RNA
 ↓ translation
Protein
```

# 13. What We Learned

이번 Notebook의 핵심 흐름:

```text
duplicated sequence classes
        ↓
common responsibility 발견
        ↓
SequenceMolecule abstraction
        ↓
subclass-specific alphabet validation
        ↓
NucleicAcid abstraction
        ↓
clean biological hierarchy
```

가장 중요한 설계 원칙은 다음입니다.

> 공통 behavior는 상위 abstraction으로 올리고,
> biological rule은 가장 적절한 domain class가 책임진다.

# 14. Exercises

## Exercise 1
`SequenceMolecule`에 `is_empty()` method를 추가해 보십시오.

그 후 이 method가 정말 필요한 API인지도 생각해 보십시오.

## Exercise 2
DNA에 `valid_codes = set("ATGC")` 대신 다른 alphabet을 넣으면 어떤 일이 생기는지 확인하십시오.

## Exercise 3
RNA와 DNA의 공통 alphabet과 다른 alphabet을 각각 계산해 보십시오.

예:

```python
DNA.valid_codes & RNA.valid_codes
DNA.valid_codes - RNA.valid_codes
RNA.valid_codes - DNA.valid_codes
```

## Exercise 4
`Protein`에 lowercase sequence와 whitespace가 포함된 sequence를 입력해 normalization 결과를 확인하십시오.

## Exercise 5 — Design Exercise
다음 기능을 어느 class에 두는 것이 좋은지 이유와 함께 정하십시오.

- DNA complement
- GC content
- protein molecular weight
- motif search

정답을 지금 확정하지 마십시오.

Notebook 03과 Notebook 04에서 실제 architecture decision을 내립니다.